# Chapter 9: Conics and Their Duals

**Source span.** Sections 9.1-9.6, printed pages 145-166, PDF pages 167-188 of *Perspectives on Projective Geometry*. The source span was read for structure, terminology, theorem targets, and examples; the prose, code, diagrams, and checks here are original.

**Chapter goal.** Build one computational dictionary for conics: a symmetric matrix gives the point equation, sends points to polar lines, produces tangents, transforms predictably under homographies, degenerates by rank and signature, and pairs with a dual matrix through a simple product condition.

A conic in this notebook is not just a curve in an affine picture. It is a projective object represented by a homogeneous quadratic form. The affine drawings are inspection windows: they let us see which matrix operation is being tested. Whenever the drawing can mislead, such as at infinity or at rank loss, the nearby code checks the algebra.


## Computational Translation Guide

| Source idea | Computational object used here | Test to keep in view |
| --- | --- | --- |
| Point in the projective plane | nonzero vector `p = [x, y, z]`, up to scale | scale changes do not change incidence or conic membership |
| Line | vector `l = [u, v, w]` with incidence `p.T @ l = 0` | joins and meets are cross products |
| Conic equation | symmetric matrix `A` with `p.T @ A @ p = 0` | the skew part of a matrix vanishes from the quadratic form |
| Polar of a point | line `A @ p` when `A` is symmetric | `p` lies on its polar exactly when `p` lies on the conic |
| Tangent at a conic point | the polar line `A @ p` for `p.T @ A @ p = 0` | line-conic intersection has a double root |
| Dual conic | line equation `l.T @ adj(A) @ l = 0` | sampled tangents have zero dual residual |
| Projective transform | for `p' = T p`, `A' = T^{-T} A T^{-1}` | transformed samples satisfy the transformed equation |
| Degenerate conic | rank-deficient `A`, classified by signature | the adjugate may retain the tangent pencil, or vanish for a double line |
| Primal-dual pair | symmetric nonzero `(A, B)` with `A @ B = lambda I` | nondegenerate cases use inverses; degenerate cases force incidence through kernels |

**Library routing.** Matplotlib is used for durable 2D projective diagrams and labeled degeneracy panels. Plotly is used where a homography slider makes the transform law inspectable without notebook widgets. SymPy supplies exact adjugate identities. NetworkX is used for the proof dependency graph of primal-dual pairs, where the object being visualized is a logical dependency structure rather than a geometric curve.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

START = Path.cwd().resolve()
roots = [START, *START.parents]
roots += [root / "Perspectives-on-Projective-Geometry" for root in roots]
BOOK_ROOT = None
for candidate in roots:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate.resolve()
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, book_relative, display_artifact, image_stats, save_json, save_table
from utils.conics import ellipse_conic, evaluate_conic, polar_line
from utils.projective import affine, hpoint, incidence, join

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-09-conics-and-their-duals"
for subdir in ["figures", "html", "tables", "checks"]:
    (ARTIFACT_ROOT / subdir).mkdir(parents=True, exist_ok=True)

artifact_paths: list[Path] = []
all_checks: dict[str, dict] = {}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "font.size": 9})
BOOK_ROOT, ARTIFACT_ROOT


In [ ]:
def rel(path: Path) -> str:
    return book_relative(path)

def save_figure(fig, filename: str) -> Path:
    path = ARTIFACT_ROOT / "figures" / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    artifact_paths.append(path)
    return path

def save_check(data: dict, filename: str) -> Path:
    return save_json(data, ARTIFACT_ROOT, "checks", filename)

def adjugate_np(matrix: np.ndarray) -> np.ndarray:
    mat = np.asarray(matrix, dtype=float)
    out = np.zeros_like(mat)
    for i in range(3):
        for j in range(3):
            minor = np.delete(np.delete(mat, i, axis=0), j, axis=1)
            out[j, i] = ((-1) ** (i + j)) * np.linalg.det(minor)
    return out

def symmetrize(matrix: np.ndarray) -> np.ndarray:
    mat = np.asarray(matrix, dtype=float)
    return 0.5 * (mat + mat.T)

def rank_num(matrix: np.ndarray, tol: float = 1e-10) -> int:
    return int(np.linalg.matrix_rank(np.asarray(matrix, dtype=float), tol=tol))

def signature_label(matrix: np.ndarray, tol: float = 1e-9) -> str:
    vals = np.linalg.eigvalsh(np.asarray(matrix, dtype=float))
    plus = int(np.sum(vals > tol))
    minus = int(np.sum(vals < -tol))
    zero = 3 - plus - minus
    # A and -A define the same conic, so use the projective convention with the larger sign count positive.
    if minus > plus:
        plus, minus = minus, plus
    return "(" + ",".join(["+"] * plus + ["-"] * minus + ["0"] * zero) + ")"

def projective_normalize(vector: np.ndarray, tol: float = 1e-12) -> np.ndarray:
    arr = np.asarray(vector, dtype=float)
    idx = int(np.argmax(np.abs(arr)))
    return arr if abs(arr[idx]) < tol else arr / arr[idx]

def projective_residual(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(projective_normalize(a) - projective_normalize(b)))

def conic_grid(A: np.ndarray, limit: float = 3.0, samples: int = 420):
    xs = np.linspace(-limit, limit, samples)
    ys = np.linspace(-limit, limit, samples)
    xx, yy = np.meshgrid(xs, ys)
    val = (
        A[0, 0] * xx * xx + A[1, 1] * yy * yy + A[2, 2]
        + 2 * A[0, 1] * xx * yy + 2 * A[0, 2] * xx + 2 * A[1, 2] * yy
    )
    return xx, yy, val

def plot_conic_contour(ax, A: np.ndarray, limit: float = 3.0, color: str = "#1f77b4", linewidth: float = 2.0):
    xx, yy, val = conic_grid(A, limit=limit)
    if float(np.nanmin(val)) <= 0 <= float(np.nanmax(val)):
        ax.contour(xx, yy, val, levels=[0], colors=[color], linewidths=linewidth)

def style_affine_axis(ax, title: str, limit: float = 3.0):
    ax.axhline(0, color="#d0d0d0", lw=0.8, zorder=0)
    ax.axvline(0, color="#d0d0d0", lw=0.8, zorder=0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_title(title)
    ax.grid(True, color="#eeeeee", lw=0.6)

def line_segment(line: np.ndarray, xlim=(-3, 3), ylim=(-3, 3), tol: float = 1e-9):
    a, b, c = np.asarray(line, dtype=float)
    pts = []
    for x in xlim:
        if abs(b) > tol:
            y = -(a * x + c) / b
            if ylim[0] - tol <= y <= ylim[1] + tol:
                pts.append((float(x), float(y)))
    for y in ylim:
        if abs(a) > tol:
            x = -(b * y + c) / a
            if xlim[0] - tol <= x <= xlim[1] + tol:
                pts.append((float(x), float(y)))
    unique = []
    for pt in pts:
        if not any(np.linalg.norm(np.array(pt) - np.array(old)) < 1e-7 for old in unique):
            unique.append(pt)
    return None if len(unique) < 2 else (unique[0], unique[1])

def plot_line(ax, line: np.ndarray, *, color="#444444", lw=1.5, ls="-", limit=3.0, alpha=1.0):
    seg = line_segment(line, xlim=(-limit, limit), ylim=(-limit, limit))
    if seg is not None:
        (x0, y0), (x1, y1) = seg
        ax.plot([x0, x1], [y0, y1], color=color, lw=lw, ls=ls, alpha=alpha)

def line_affine_basis(line: np.ndarray, tol: float = 1e-10):
    a, b, c = np.asarray(line, dtype=float)
    if abs(a) < tol and abs(b) < tol:
        raise ValueError("line at infinity in this affine chart")
    point = np.array([0.0, -c / b, 1.0]) if abs(b) > abs(a) else np.array([-c / a, 0.0, 1.0])
    return point, np.array([-b, a, 0.0])

def line_conic_coefficients(A: np.ndarray, line: np.ndarray):
    p0, d = line_affine_basis(line)
    return float(d @ A @ d), float(2 * p0 @ A @ d), float(p0 @ A @ p0)

def line_conic_discriminant(A: np.ndarray, line: np.ndarray) -> float:
    aa, bb, cc = line_conic_coefficients(A, line)
    return float(bb * bb - 4 * aa * cc)

def real_line_conic_intersections(A: np.ndarray, line: np.ndarray, tol: float = 1e-8) -> list[np.ndarray]:
    p0, d = line_affine_basis(line)
    aa, bb, cc = line_conic_coefficients(A, line)
    roots = np.roots([aa, bb, cc]) if abs(aa) > tol else np.roots([bb, cc])
    return [p0 + float(root.real) * d for root in roots if abs(root.imag) < 1e-7]


## 1. Conic Equations As Symmetric Matrices

Homogenizing the circle equation gives a quadratic equation in `(x, y, z)`, but the real payload is the matrix form `p.T @ A @ p = 0`. The matrix may be scaled without changing the projective conic, and only its symmetric part contributes to the quadratic form.

The figure compares ordinary affine views with the projective signature data. The first row distinguishes ellipse, parabola, and hyperbola by the conic's intersection with the line at infinity. The second row shows rank-deficient models: two real lines, two complex-conjugate lines with one real meeting point, and a double line.


In [ ]:
A_ellipse = np.diag([1 / 2.1**2, 1 / 1.1**2, -1.0])
A_parabola = np.array([[0.0, 0.0, -0.5], [0.0, 1.0, 0.0], [-0.5, 0.0, 0.0]])
A_hyperbola = np.diag([1.0, -1.0, -1.0])
A_two_real_lines = np.diag([1.0, -1.0, 0.0])
A_complex_lines = np.diag([1.0, 1.0, 0.0])
A_double_line = np.diag([1.0, 0.0, 0.0])
models = [
    ("ellipse", A_ellipse, "no real infinity point"),
    ("parabola", A_parabola, "tangent to infinity"),
    ("hyperbola", A_hyperbola, "two infinity points"),
    ("two real lines", A_two_real_lines, "rank 2, real factors"),
    ("complex line pair", A_complex_lines, "only real meeting point"),
    ("double line", A_double_line, "rank 1"),
]

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, (name, A, note) in zip(axes.flat, models):
    style_affine_axis(ax, f"{name}\nrank {rank_num(A)}, signature {signature_label(A)}", limit=3.0)
    if name == "two real lines":
        xs = np.linspace(-3, 3, 2)
        ax.plot(xs, xs, color="#1f77b4", lw=2)
        ax.plot(xs, -xs, color="#1f77b4", lw=2)
        ax.scatter([0], [0], color="#d62728", zorder=5)
    elif name == "complex line pair":
        ax.scatter([0], [0], color="#d62728", zorder=5)
        ax.text(-2.7, 2.25, "real locus is one point", color="#444444")
        ax.add_patch(plt.Circle((0, 0), 1.2, fill=False, color="#aaaaaa", ls="--"))
    elif name == "double line":
        ax.axvline(0, color="#1f77b4", lw=5, alpha=0.35)
        ax.axvline(0, color="#1f77b4", lw=1.5)
    else:
        plot_conic_contour(ax, A, limit=3.0)
    ax.text(-2.9, -2.75, f"det A_xy = {np.linalg.det(A[:2, :2]):.3g}\n{note}", fontsize=8)
fig.suptitle("Matrix conics: real affine views of projective signatures", y=1.02, fontsize=12)
taxonomy_path = save_figure(fig, "conic-matrix-signature-taxonomy.png")

raw = np.array([[2.0, 5.0, -1.0], [-3.0, 1.0, 4.0], [0.5, -2.0, -1.5]])
raw_sym = symmetrize(raw)
sample_points = [hpoint(-1.2, 0.7), hpoint(0.4, -1.1), hpoint(2.0, 0.25, 0.8)]
matrix_checks = {
    "symmetrization_residual": float(max(abs(p @ raw @ p - p @ raw_sym @ p) for p in sample_points)),
    "models": [
        {
            "model": name,
            "rank": rank_num(A),
            "signature": signature_label(A),
            "det_upper_left_block": float(np.linalg.det(A[:2, :2])),
            "determinant": float(np.linalg.det(A)),
            "interpretation": note,
        }
        for name, A, note in models
    ],
    "ellipse_parabola_hyperbola_block_signs": {
        "ellipse": float(np.sign(np.linalg.det(A_ellipse[:2, :2]))),
        "parabola": float(np.sign(np.linalg.det(A_parabola[:2, :2]))),
        "hyperbola": float(np.sign(np.linalg.det(A_hyperbola[:2, :2]))),
    },
}
all_checks["matrix_conic_signature"] = matrix_checks
save_check(matrix_checks, "matrix-conic-signature-checks.json")
display_artifact(taxonomy_path, width=820)


## 2. Polars And Tangents

For a nondegenerate symmetric conic matrix `A`, polarity is the map `p -> A p` from points to lines. If `p` is on the conic, the polar is the tangent. If `p` is outside an ellipse, the polar cuts the conic at the two tangency points of the tangents drawn from `p`. If `p` is inside, the polar misses the real conic.

The discriminant of the line-conic intersection records the same inside/on/outside distinction that the picture suggests.


In [ ]:
rx, ry = 1.75, 1.05
A_polar = ellipse_conic(rx=rx, ry=ry)
inside_point = hpoint(0.35, 0.2)
on_point = hpoint(rx * math.cos(0.78), ry * math.sin(0.78))
outside_point = hpoint(2.55, 1.15)
cases = [("inside", inside_point, "#9467bd"), ("on conic", on_point, "#2ca02c"), ("outside", outside_point, "#d62728")]

fig, ax = plt.subplots(figsize=(8.2, 6.4))
style_affine_axis(ax, "Polarity dictionary for an ellipse", limit=3.2)
plot_conic_contour(ax, A_polar, limit=3.2, color="#1f77b4", linewidth=2.2)
polar_lines = {}
polar_discriminants = {}
for label, point, color in cases:
    xy = affine(point)
    ax.scatter([xy[0]], [xy[1]], color=color, s=35, zorder=6)
    ax.text(xy[0] + 0.06, xy[1] + 0.08, label, color=color)
    line = polar_line(A_polar, point)
    polar_lines[label] = line
    plot_line(ax, line, color=color, lw=1.7, ls="--" if label != "on conic" else "-", limit=3.2)
    polar_discriminants[label] = line_conic_discriminant(A_polar, line)

outside_tangent_points = real_line_conic_intersections(A_polar, polar_lines["outside"])
for tpoint in outside_tangent_points:
    txy = affine(tpoint)
    ax.scatter([txy[0]], [txy[1]], color="#d62728", s=25, zorder=6)
    plot_line(ax, join(outside_point, tpoint), color="#d62728", lw=1.0, alpha=0.75, limit=3.2)
ax.text(-3.0, -3.0, "red joins: tangents from outside point; dashed lines: polars", fontsize=8)
polar_path = save_figure(fig, "polar-tangent-incidence-dictionary.png")

A_inv = np.linalg.inv(A_polar)
polar_checks = {
    "on_conic_value": float(evaluate_conic(A_polar, on_point)),
    "on_point_tangent_incidence": float(on_point @ polar_lines["on conic"]),
    "inside_polar_discriminant": float(polar_discriminants["inside"]),
    "on_polar_discriminant": float(polar_discriminants["on conic"]),
    "outside_polar_discriminant": float(polar_discriminants["outside"]),
    "outside_tangent_point_values": [float(t @ A_polar @ t) for t in outside_tangent_points],
    "outside_tangent_lines_pass_source": [bool(incidence(outside_point, join(outside_point, t))) for t in outside_tangent_points],
    "polarity_involution_max_residual": float(max(projective_residual(A_inv @ polar_line(A_polar, p), p) for _, p, _ in cases)),
}
all_checks["polar_tangent"] = polar_checks
save_check(polar_checks, "polar-tangent-incidence-checks.json")
display_artifact(polar_path, width=760)
polar_checks


## 3. Dual Quadratic Forms

The dual conic is a set of lines. For a nondegenerate conic `p.T @ A @ p = 0`, each tangent has the form `l = A p`, and those line coordinates satisfy `l.T @ adj(A) @ l = 0`. The right panel uses the line-space chart `y = m x + b`, or `[m, -1, b]`; vertical tangents are outside this chart.


In [ ]:
adj_polar = adjugate_np(A_polar)
thetas = np.linspace(0, 2 * np.pi, 48, endpoint=False)
conic_points = [hpoint(rx * math.cos(t), ry * math.sin(t)) for t in thetas]
tangent_lines = [polar_line(A_polar, p) for p in conic_points]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.8))
style_affine_axis(ax0, "Primal conic with sampled tangents", limit=3.0)
plot_conic_contour(ax0, A_polar, limit=3.0, color="#1f77b4")
for line in tangent_lines[::2]:
    plot_line(ax0, line, color="#2ca02c", lw=0.8, alpha=0.45, limit=3.0)
ax0.text(-2.8, -2.75, "Each green line is A p", fontsize=8)

m_vals, b_vals = [], []
for u, v, w in tangent_lines:
    if abs(v) > 1e-7:
        m_vals.append(float(-u / v))
        b_vals.append(float(-w / v))
ms = np.linspace(-4.0, 4.0, 360)
bs = np.linspace(-2.8, 2.8, 360)
MM, BB = np.meshgrid(ms, bs)
L = np.stack([MM, -np.ones_like(MM), BB], axis=-1)
dual_values = np.einsum("...i,ij,...j->...", L, adj_polar, L)
ax1.contour(MM, BB, dual_values, levels=[0], colors=["#1f77b4"], linewidths=2.0)
ax1.scatter(m_vals, b_vals, color="#2ca02c", s=18, alpha=0.8)
ax1.grid(True, color="#eeeeee", lw=0.6)
ax1.set_xlabel("slope m")
ax1.set_ylabel("intercept b")
ax1.set_title("Dual conic in line chart [m, -1, b]")
dual_path = save_figure(fig, "dual-conic-tangent-envelope.png")

dual_checks = {
    "max_tangent_dual_residual": float(max(abs(line @ adj_polar @ line) for line in tangent_lines)),
    "adjugate_identity_residual": float(np.linalg.norm(A_polar @ adj_polar - np.linalg.det(A_polar) * np.eye(3))),
    "sampled_tangent_count": len(tangent_lines),
    "line_chart_points_used": len(m_vals),
}
all_checks["dual_conic"] = dual_checks
save_check(dual_checks, "dual-conic-envelope-checks.json")
display_artifact(dual_path, width=850)
dual_checks


## 4. How Conics Transform

If a projective transformation sends points by `p' = T p`, the transformed conic matrix is not `T A T.T`. The equation must be pulled back from the new point to the old point:

`A' = T^{-T} A T^{-1}`.

Dual conics transform in the opposite direction: `B' = T B T.T`.


In [ ]:
A_circle = np.diag([1.0, 1.0, -1.0])
T = np.array([[1.18, 0.32, 0.18], [-0.18, 0.92, 0.28], [0.16, -0.10, 1.0]])
T_inv = np.linalg.inv(T)
A_transformed = T_inv.T @ A_circle @ T_inv
circle_thetas = np.linspace(0, 2 * np.pi, 300)
circle_points = np.column_stack([np.cos(circle_thetas), np.sin(circle_thetas), np.ones_like(circle_thetas)])
transformed_h = (T @ circle_points.T).T
transformed_xy = np.array([affine(p) for p in transformed_h])

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.8, 4.8))
style_affine_axis(ax0, "Original unit circle", limit=2.6)
plot_conic_contour(ax0, A_circle, limit=2.6)
ax0.scatter(circle_points[::24, 0], circle_points[::24, 1], s=12, color="#2ca02c")
style_affine_axis(ax1, "Image under p' = T p", limit=3.0)
plot_conic_contour(ax1, A_transformed, limit=3.0)
ax1.plot(transformed_xy[:, 0], transformed_xy[:, 1], color="#2ca02c", lw=1.2)
ax1.scatter(transformed_xy[::24, 0], transformed_xy[::24, 1], s=12, color="#d62728")
ax1.text(-2.85, -2.72, "samples satisfy q.T A' q = 0", fontsize=8)
transform_path = save_figure(fig, "projective-conic-transform-law.png")

transform_checks = {
    "det_T": float(np.linalg.det(T)),
    "max_transformed_point_residual": float(max(abs(q @ A_transformed @ q) for q in transformed_h)),
    "original_signature": signature_label(A_circle),
    "transformed_signature": signature_label(A_transformed),
    "dual_transform_product_residual": float(np.linalg.norm((T_inv.T @ A_circle @ T_inv) @ (T @ np.linalg.inv(A_circle) @ T.T) - np.eye(3))),
}

frames = []
for t in np.linspace(0, 1, 13):
    Tt = (1 - t) * np.eye(3) + t * T
    pts_h = (Tt @ circle_points.T).T
    pts = np.array([affine(p) for p in pts_h])
    frames.append(go.Frame(data=[go.Scatter(x=pts[:, 0], y=pts[:, 1], mode="lines", line=dict(color="#1f77b4", width=3))], name=f"{t:.2f}"))
initial_xy = circle_points[:, :2]
fig_html = go.Figure(data=[go.Scatter(x=initial_xy[:, 0], y=initial_xy[:, 1], mode="lines", line=dict(color="#1f77b4", width=3))], frames=frames)
steps = [dict(method="animate", args=[[fr.name], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}], label=fr.name) for fr in frames]
fig_html.update_layout(
    title="Projective conic transform lab: T_t from identity to T",
    xaxis=dict(range=[-3, 3], zeroline=True, scaleanchor="y", scaleratio=1),
    yaxis=dict(range=[-3, 3], zeroline=True),
    sliders=[dict(active=0, steps=steps, currentvalue={"prefix": "t = "})],
    updatemenus=[dict(type="buttons", showactive=False, buttons=[dict(label="play", method="animate", args=[None, {"frame": {"duration": 180, "redraw": True}, "fromcurrent": True}])])],
    width=780,
    height=660,
)
html_path = ARTIFACT_ROOT / "html" / "projective-conic-transform-lab.html"
fig_html.write_html(str(html_path), include_plotlyjs=True, full_html=True)
artifact_paths.append(html_path)

all_checks["projective_transform"] = transform_checks
save_check(transform_checks, "projective-transform-law-checks.json")
display_artifact(transform_path, width=820)
display_artifact(html_path, width="100%", height=520)
transform_checks


## 5. Degenerate Conics And What The Adjugate Keeps

Degenerate conics are part of the projective story, not errors to discard. A rank-two conic can factor as two distinct lines `g` and `h`; the adjugate records the singular point through

`adj(g h.T + h g.T) = -(g x h)(g x h).T`.

A double line is rank one, and its adjugate vanishes. That is why the chapter introduces primal-dual pairs later: the primal matrix alone may no longer carry the limiting tangent data.


In [ ]:
alphas = [0.55, 0.18, 0.0, -0.18, -0.55]
fig, axes = plt.subplots(1, len(alphas), figsize=(13, 3.1), sharex=True, sharey=True)
for ax, alpha in zip(axes, alphas):
    style_affine_axis(ax, f"alpha = {alpha:g}", limit=2.2)
    A_alpha = np.diag([alpha, 1.0, -alpha])
    if abs(alpha) < 1e-12:
        ax.axhline(0, color="#1f77b4", lw=5, alpha=0.35)
        ax.axhline(0, color="#1f77b4", lw=1.5)
        ax.text(-1.85, 1.65, "double line\ny = 0", fontsize=8)
    else:
        plot_conic_contour(ax, A_alpha, limit=2.2)
    ax.text(-2.0, -1.95, f"rank {rank_num(A_alpha)}\n{signature_label(A_alpha)}", fontsize=8)
fig.suptitle("Deformation alpha x^2 + y^2 - alpha z^2 = 0 through a double line", y=1.08, fontsize=12)
degenerate_limit_path = save_figure(fig, "degenerate-conic-signature-limits.png")

g = np.array([1.0, -1.0, 0.0])
h = np.array([1.0, 1.0, 0.0])
A_pair = np.outer(g, h) + np.outer(h, g)
singular_point = np.cross(g, h)
adj_pair = adjugate_np(A_pair)
expected_adj_pair = -np.outer(singular_point, singular_point)

fig, ax = plt.subplots(figsize=(5.8, 5.8))
style_affine_axis(ax, "Two-line conic and dual tangent pencil", limit=2.6)
plot_line(ax, g, color="#1f77b4", lw=2.3, limit=2.6)
plot_line(ax, h, color="#1f77b4", lw=2.3, limit=2.6)
for angle in np.linspace(0, np.pi, 12, endpoint=False):
    plot_line(ax, np.array([math.sin(angle), -math.cos(angle), 0.0]), color="#2ca02c", lw=0.8, alpha=0.45, limit=2.6)
ax.scatter([0], [0], color="#d62728", s=35, zorder=5)
ax.text(0.1, 0.12, "g x h", color="#d62728")
ax.text(-2.45, -2.35, "green lines satisfy l.T adj(A) l = 0", fontsize=8)
tangent_pencil_path = save_figure(fig, "two-line-dual-tangent-pencil.png")

pencil_lines = [np.array([math.sin(a), -math.cos(a), 0.0]) for a in np.linspace(0, np.pi, 12, endpoint=False)]
degenerate_checks = {
    "line_pair_signature": signature_label(A_pair),
    "line_pair_rank": rank_num(A_pair),
    "singular_point": singular_point.tolist(),
    "line_pair_adjugate_residual": float(np.linalg.norm(adj_pair - expected_adj_pair)),
    "max_tangent_pencil_dual_residual": float(max(abs(line @ adj_pair @ line) for line in pencil_lines)),
    "double_line_adjugate_norm": float(np.linalg.norm(adjugate_np(A_double_line))),
    "deformation_ranks": {str(alpha): rank_num(np.diag([alpha, 1.0, -alpha])) for alpha in alphas},
    "deformation_signatures": {str(alpha): signature_label(np.diag([alpha, 1.0, -alpha])) for alpha in alphas},
}
all_checks["degenerate_conics"] = degenerate_checks
save_check(degenerate_checks, "degenerate-adjugate-checks.json")
display_artifact(degenerate_limit_path, width=900)
display_artifact(tangent_pencil_path, width=560)
degenerate_checks


## 6. Primal-Dual Pairs

The last section stops pretending that one matrix always carries all information. A conic can be represented by a primal matrix `A` and a dual matrix `B`, linked by `A @ B = lambda I`.

If `A` is invertible, `B` is a nonzero multiple of `A^{-1}`. If `A` has rank two, the product must be zero and `B` collapses to the singular point. If `A` has rank one, `B` must live on the double line described by `A`. The graph is a proof scaffold, and the CSV lists concrete representatives.


In [ ]:
def pair_residual(A: np.ndarray, B: np.ndarray) -> tuple[float, float]:
    product = A @ B
    scalar = float(np.trace(product) / 3.0)
    return scalar, float(np.linalg.norm(product - scalar * np.eye(3)))

line_l = np.array([1.0, 0.0, 0.0])
p_real = np.array([0.0, 1.0, 1.0])
q_real = np.array([0.0, -1.0, 1.0])
B_two_real_points = np.outer(p_real, q_real) + np.outer(q_real, p_real)
B_two_complex_points = np.diag([0.0, 1.0, 1.0])
B_double_point = np.outer(np.array([0.0, 0.0, 1.0]), np.array([0.0, 0.0, 1.0]))
pair_models = [
    ("real nondegenerate", np.diag([1.0, 1.0, -1.0]), np.diag([1.0, 1.0, -1.0]), "inverse pair"),
    ("complex nondegenerate", np.eye(3), np.eye(3), "inverse pair with no real point locus"),
    ("two real lines plus double point", np.diag([1.0, -1.0, 0.0]), B_double_point, "rank 2 primal, rank 1 dual"),
    ("two complex lines plus double point", np.diag([1.0, 1.0, 0.0]), B_double_point, "rank 2 primal, rank 1 dual"),
    ("double line through two real points", np.outer(line_l, line_l), B_two_real_points, "rank 1 primal, rank 2 dual"),
    ("double line through two complex points", np.outer(line_l, line_l), B_two_complex_points, "rank 1 primal, rank 2 dual"),
    ("double line with double point on it", np.outer(line_l, line_l), B_double_point, "rank 1 primal, rank 1 dual"),
]
pair_rows, pair_residuals = [], []
for name, A, B, reading in pair_models:
    scalar, residual = pair_residual(A, B)
    pair_residuals.append(residual)
    pair_rows.append({
        "model": name,
        "primal_signature": signature_label(A),
        "dual_signature": signature_label(B),
        "rank_A": rank_num(A),
        "rank_B": rank_num(B),
        "lambda_from_trace": scalar,
        "AB_minus_lambda_I_norm": residual,
        "geometric_reading": reading,
    })
pair_table_path = save_table(pair_rows, ARTIFACT_ROOT, "tables", "primal-dual-pair-signatures.csv")
artifact_paths.append(pair_table_path)

G = nx.DiGraph()
G.add_edges_from([
    ("symmetric A", "rank(A)=3"), ("symmetric A", "rank(A)=2"), ("symmetric A", "rank(A)=1"),
    ("rank(A)=3", "B scalar A^{-1}"), ("rank(A)=2", "A B = 0"),
    ("A B = 0", "columns of B in ker(A)"), ("columns of B in ker(A)", "B = p p^T at singular point"),
    ("rank(A)=1", "A = l l^T"), ("A = l l^T", "columns of B lie on l"),
    ("columns of B lie on l", "rank(B)=2: two points"), ("columns of B lie on l", "rank(B)=1: double point"),
    ("B scalar A^{-1}", "A B = lambda I"), ("B = p p^T at singular point", "A B = lambda I"),
    ("rank(B)=2: two points", "A B = lambda I"), ("rank(B)=1: double point", "A B = lambda I"),
])
pos = {
    "symmetric A": (0, 3), "rank(A)=3": (-3, 2), "rank(A)=2": (0, 2), "rank(A)=1": (3, 2),
    "B scalar A^{-1}": (-3, 1), "A B = 0": (0, 1.2), "columns of B in ker(A)": (0, 0.4),
    "B = p p^T at singular point": (0, -0.4), "A = l l^T": (3, 1.2),
    "columns of B lie on l": (3, 0.4), "rank(B)=2: two points": (2.1, -0.4),
    "rank(B)=1: double point": (4.0, -0.4), "A B = lambda I": (0.5, -1.6),
}
fig, ax = plt.subplots(figsize=(12, 6.2))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=12, edge_color="#777777")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#f7f7f7", edgecolors="#1f77b4", node_size=1900, linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_axis_off()
ax.set_title("Proof scaffold for primal-dual conic pairs")
pair_graph_path = save_figure(fig, "primal-dual-pair-admissibility-graph.png")

pair_checks = {"pair_count": len(pair_rows), "max_AB_minus_lambda_I_norm": float(max(pair_residuals)), "rows": pair_rows}
all_checks["primal_dual_pairs"] = pair_checks
save_check(pair_checks, "primal-dual-pair-checks.json")
display_artifact(pair_graph_path, width=900)
display_artifact(pair_table_path)
pd.DataFrame(pair_rows)


## Artifact Gallery

![Conic matrix signature taxonomy](../../artifacts/chapter-09-conics-and-their-duals/figures/conic-matrix-signature-taxonomy.png)

![Polar tangent incidence dictionary](../../artifacts/chapter-09-conics-and-their-duals/figures/polar-tangent-incidence-dictionary.png)

![Dual conic tangent envelope](../../artifacts/chapter-09-conics-and-their-duals/figures/dual-conic-tangent-envelope.png)

![Projective conic transform law](../../artifacts/chapter-09-conics-and-their-duals/figures/projective-conic-transform-law.png)

![Degenerate conic signature limits](../../artifacts/chapter-09-conics-and-their-duals/figures/degenerate-conic-signature-limits.png)

![Two-line dual tangent pencil](../../artifacts/chapter-09-conics-and-their-duals/figures/two-line-dual-tangent-pencil.png)

![Primal-dual pair admissibility graph](../../artifacts/chapter-09-conics-and-their-duals/figures/primal-dual-pair-admissibility-graph.png)

Open the interactive [projective conic transform lab](../../artifacts/chapter-09-conics-and-their-duals/html/projective-conic-transform-lab.html), the [primal-dual pair table](../../artifacts/chapter-09-conics-and-their-duals/tables/primal-dual-pair-signatures.csv), and the [deformation sweep table](../../artifacts/chapter-09-conics-and-their-duals/tables/deformation-sweep-conic-duality.csv) when reading outside an executed kernel.


## Applied Lab: Move Through A Double Line

Use `A(alpha) = diag(alpha, 1, -alpha)` as a small laboratory. For positive `alpha` the affine picture is an ellipse squeezed toward the x-axis. At `alpha = 0` it is the double line `y = 0`. For negative `alpha` the curve reopens as a hyperbola. The table records what the picture alone cannot: rank, signature, determinant, and adjugate rank.

Change the alpha list and re-execute the next cell. The expected invariant is not the visible shape; it is the way rank and adjugate rank announce which kind of dual information remains.


In [ ]:
sweep_alphas = [-0.75, -0.25, -0.05, 0.0, 0.05, 0.25, 0.75]
sweep_rows = []
for alpha in sweep_alphas:
    A = np.diag([alpha, 1.0, -alpha])
    adjA = adjugate_np(A)
    sweep_rows.append({
        "alpha": alpha,
        "rank_A": rank_num(A),
        "signature_A": signature_label(A),
        "det_A": float(np.linalg.det(A)),
        "rank_adjugate_A": rank_num(adjA),
        "adjugate_norm": float(np.linalg.norm(adjA)),
        "affine_reading": "double line" if abs(alpha) < 1e-12 else ("ellipse" if alpha > 0 else "hyperbola"),
    })
sweep_table_path = save_table(sweep_rows, ARTIFACT_ROOT, "tables", "deformation-sweep-conic-duality.csv")
artifact_paths.append(sweep_table_path)
sweep_checks = {
    "zero_alpha_rank": [row for row in sweep_rows if row["alpha"] == 0.0][0]["rank_A"],
    "zero_alpha_adjugate_norm": [row for row in sweep_rows if row["alpha"] == 0.0][0]["adjugate_norm"],
    "nonzero_alpha_rank_min": int(min(row["rank_A"] for row in sweep_rows if row["alpha"] != 0.0)),
    "sweep_row_count": len(sweep_rows),
}
all_checks["deformation_sweep"] = sweep_checks
save_check(sweep_checks, "deformation-sweep-checks.json")
display_artifact(sweep_table_path)
pd.DataFrame(sweep_rows)


## Final Sanity Checks And Takeaways

The notebook has used five matrix facts as its backbone:

- The quadratic form only sees the symmetric part of a matrix.
- A point on a nondegenerate conic is incident with its polar, which is the tangent.
- Tangent line coordinates satisfy the dual quadratic form using the adjugate.
- Projective transformations pull primal conic matrices back by the inverse matrix.
- Degenerate conics need primal-dual pairs because the adjugate may either record a singular point or vanish.

The final cell checks those claims, verifies artifact integrity, and writes the chapter sanity record.


In [ ]:
a, b, c, d, e, f = sp.symbols("a b c d e f")
A_sym = sp.Matrix([[a, b, d], [b, c, e], [d, e, f]])
adj_identity_symbolic = sp.simplify(A_sym * A_sym.adjugate() - A_sym.det() * sp.eye(3)) == sp.zeros(3)
g_sym = sp.Matrix([1, -1, 0])
h_sym = sp.Matrix([1, 1, 0])
A_pair_sym = g_sym * h_sym.T + h_sym * g_sym.T
p_sym = g_sym.cross(h_sym)
line_pair_adjugate_symbolic = sp.simplify(A_pair_sym.adjugate() + p_sym * p_sym.T) == sp.zeros(3)

line_samples = np.array([-1.4, -0.2, 0.75, 1.6])
def cr(vals):
    a0, b0, c0, d0 = vals
    return ((a0 - c0) * (b0 - d0)) / ((a0 - d0) * (b0 - c0))
line_image = np.array([(1.15 * x - 0.2) / (0.18 * x + 1.0) for x in line_samples])
cross_ratio_error = float(abs(cr(line_samples) - cr(line_image)))

all_artifacts = list(dict.fromkeys(artifact_paths))
assert_artifacts(all_artifacts)
raster_stats = [image_stats(path) for path in all_artifacts if path.suffix.lower() in {".png", ".jpg", ".jpeg"}]
for stats in raster_stats:
    assert stats["pixel_std"] > 1.0, stats

assert matrix_checks["symmetrization_residual"] < 1e-10
assert abs(polar_checks["on_conic_value"]) < 1e-10
assert abs(polar_checks["on_point_tangent_incidence"]) < 1e-10
assert polar_checks["inside_polar_discriminant"] < 0
assert abs(polar_checks["on_polar_discriminant"]) < 1e-8
assert polar_checks["outside_polar_discriminant"] > 0
assert dual_checks["max_tangent_dual_residual"] < 1e-10
assert transform_checks["max_transformed_point_residual"] < 1e-10
assert transform_checks["original_signature"] == transform_checks["transformed_signature"]
assert degenerate_checks["line_pair_adjugate_residual"] < 1e-10
assert degenerate_checks["double_line_adjugate_norm"] < 1e-12
assert pair_checks["max_AB_minus_lambda_I_norm"] < 1e-10
assert sweep_checks["zero_alpha_rank"] == 1
assert sweep_checks["zero_alpha_adjugate_norm"] < 1e-12
assert adj_identity_symbolic and line_pair_adjugate_symbolic
assert cross_ratio_error < 1e-12

final = {
    "chapter": 9,
    "source_span": "printed pp. 145-166 / PDF pp. 167-188; sections 9.1-9.6",
    "all_files_exist": all(path.exists() and path.stat().st_size > 256 for path in all_artifacts),
    "artifacts": [rel(path) for path in all_artifacts],
    "checks": all_checks,
    "raster_artifacts": [{**stats, "path": rel(Path(stats["path"]))} for stats in raster_stats],
    "symbolic_checks": {
        "A_adjugate_identity": bool(adj_identity_symbolic),
        "two_line_adjugate_identity": bool(line_pair_adjugate_symbolic),
    },
    "cross_ratio_error": cross_ratio_error,
    "notebook_executed": True,
}
visual_checks = {
    "chapter": 9,
    "all_files_exist": final["all_files_exist"],
    "visual_count": len(all_artifacts),
    "artifacts": final["artifacts"],
    "raster_artifacts": final["raster_artifacts"],
    "cross_ratio_error": cross_ratio_error,
    "numeric_checks": {
        "polar_on_conic_value": polar_checks["on_conic_value"],
        "dual_tangent_max_residual": dual_checks["max_tangent_dual_residual"],
        "transform_max_residual": transform_checks["max_transformed_point_residual"],
        "line_pair_adjugate_residual": degenerate_checks["line_pair_adjugate_residual"],
        "pair_product_max_residual": pair_checks["max_AB_minus_lambda_I_norm"],
    },
}
save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final
